# Prototipo funcional — Control de calidad de cantidades (Gradio)
**Grupo 12** · modelo: DNN con embeddings de entidad + historial (Práctica 2, MAE ~0,43)

**Caso de uso:** el modelo no *predice* la cantidad (ya viene en el pedido), sino que la usa como **segunda opinión**: calcula la cantidad *esperada* para un cliente-producto-canal y **marca las líneas cargadas que se alejan mucho**, para revisar errores de carga (p. ej. 40 en vez de 4).

**Arquetipo: Human-in-the-Loop.** El modelo prioriza qué revisar; la persona confirma o corrige. El historial del cliente se **toma solo** (no se tipea), evitando la circularidad.

### Cómo correr
1. Instalación. 2. Subir `ventas_adventureworks.csv`. 3. App (entrena la DNN, ~1-2 min) → link público con `share=True`.

In [1]:
!pip install -q gradio tensorflow scikit-learn

## Subir el dataset

In [2]:
import glob, os
from google.colab import drive

drive.mount('/content/drive')
candidatos = glob.glob('/content/drive/MyDrive/**/ventas_adventureworks*.csv*', recursive=True)

if candidatos:
    RUTA_CSV = candidatos[0]
    print("Encontrado en Drive:", RUTA_CSV)
else:
    print("No se encontró el CSV en este Drive. Subilo manualmente (sirve .csv o .csv.gz):")
    from google.colab import files
    subida = files.upload()
    nombre_subido = next(iter(subida))
    RUTA_CSV = '/content/' + nombre_subido
    os.replace(nombre_subido, RUTA_CSV)

print('listo:', os.path.exists(RUTA_CSV), '->', RUTA_CSV)


Saving ventas_adventureworks.csv.gz to ventas_adventureworks.csv.gz
listo: True


## Prototipo (modelo P2 + control de calidad)

In [3]:
import numpy as np, pandas as pd, warnings
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import gradio as gr
warnings.filterwarnings("ignore"); tf.get_logger().setLevel("ERROR")
SEED = 42; keras.utils.set_random_seed(SEED)
NUM = ["territoryid","unitprice","unitpricediscount","listprice","standardcost","discountpct",
       "minqty","maxqty","anio","mes","dia","diasemana","semanaanio","tienemaxqty",
       "duracionoferta","diasdesdeoferta","cliente_prom_hist","cliente_frecuencia_hist",
       "tienda_prom_hist","cliente_nuevo"]
CAT_OH  = ["categoria","oferta","categoriaoferta","canalventa"]
CAT_EMB = ["producto","subcategoria"]

def construir():
    df = pd.read_csv(RUTA_CSV)  # pandas detecta solo si está comprimido, según la extensión
    for c in ["orderdate","iniciooferta","finoferta"]:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    df["anio"]=df["orderdate"].dt.year; df["mes"]=df["orderdate"].dt.month; df["dia"]=df["orderdate"].dt.day
    df["diasemana"]=df["orderdate"].dt.dayofweek; df["semanaanio"]=df["orderdate"].dt.isocalendar().week.astype(int)
    df["canalventa"]=np.select([df["onlineorderflag"].eq(1),df["storeid"].notna()],["Internet","Tienda"],default="Tienda")
    df["tienemaxqty"]=df["maxqty"].notna().astype(int); df["maxqty"]=df["maxqty"].fillna(0)
    df["duracionoferta"]=(df["finoferta"]-df["iniciooferta"]).dt.days.fillna(0)
    df["diasdesdeoferta"]=(df["orderdate"]-df["iniciooferta"]).dt.days.fillna(0)
    cc=df.groupby("orderdate").size().sort_index().cumsum(); T=cc.iloc[-1]
    ct=cc[cc>=T*0.70].index[0]; tr=df[df["orderdate"]<ct]; g=float(tr["cantidadvendida"].mean())
    hist_prom=tr.groupby("customerid")["cantidadvendida"].mean()
    hist_frec=tr.groupby("customerid").size()
    df["cliente_prom_hist"]=df["customerid"].map(hist_prom).fillna(g)
    df["cliente_frecuencia_hist"]=df["customerid"].map(hist_frec).fillna(hist_frec.median())
    df["tienda_prom_hist"]=df["storeid"].map(tr.groupby("storeid")["cantidadvendida"].mean()).fillna(g)
    df["cliente_nuevo"]=(~df["customerid"].isin(tr["customerid"].unique())).astype(int)
    for c in CAT_OH+CAT_EMB: df[c]=df[c].astype(str)
    trn=df[df["orderdate"]<ct]
    pre=ColumnTransformer([("num",StandardScaler(),NUM),
                           ("oh",OneHotEncoder(handle_unknown="ignore",sparse_output=False),CAT_OH)]).fit(trn)
    vocab={}; edim={}
    for col in CAT_EMB:
        cats=sorted(trn[col].unique()); vocab[col]={v:i+1 for i,v in enumerate(cats)}
        edim[col]=int(max(2,min(50,(len(cats)+2)//2)))
    def idx(s,col): return s[col].map(lambda v: vocab[col].get(v,0)).astype("int32").values
    Rtr=pre.transform(trn).astype("float32"); ytr=trn["cantidadvendida"].astype("float32").values
    Xtr={"producto":idx(trn,"producto"),"subcategoria":idx(trn,"subcategoria"),"resto":Rtr}
    keras.utils.set_random_seed(SEED); ins=[]; embs=[]
    for col in CAT_EMB:
        i=keras.Input(shape=(1,),name=col,dtype="int32"); ins.append(i)
        embs.append(layers.Flatten()(layers.Embedding(len(vocab[col])+1,edim[col])(i)))
    r=keras.Input(shape=(Rtr.shape[1],),name="resto"); ins.append(r)
    x=layers.Concatenate()(embs+[r])
    for u in [128,64,32,16]:
        x=layers.Dense(u)(x); x=layers.BatchNormalization()(x); x=layers.Activation("relu")(x); x=layers.Dropout(0.20)(x)
    modelo=keras.Model(ins,layers.Dense(1)(x)); modelo.compile(optimizer=keras.optimizers.Adam(5e-4),loss="mse")
    modelo.fit(Xtr,ytr,epochs=40,batch_size=512,verbose=0)
    pnum=df.groupby("producto")[NUM].median(numeric_only=True)
    pcat=df.groupby("producto")[CAT_OH].agg(lambda s:s.mode().iloc[0])
    prod2sub=df.groupby("producto")["subcategoria"].agg(lambda s:s.mode().iloc[0]).to_dict()
    hp=hist_prom.round(1)
    muestra=(pd.concat([hp.sort_values().head(120), hp.sort_values().tail(120),
                        hp.sample(min(120,len(hp)),random_state=1)]).drop_duplicates())
    clientes={f"Cliente {cid} · compra hist. ~{prom:.0f} u": (float(prom), float(hist_frec.get(cid,hist_frec.median())))
              for cid,prom in muestra.items()}
    clientes["Cliente nuevo (sin historial)"] = (g, float(hist_frec.median()))
    return dict(modelo=modelo, pre=pre, vocab=vocab, g=g, pnum=pnum, pcat=pcat, prod2sub=prod2sub,
                productos=sorted(vocab["producto"].keys()), clientes=clientes,
                cli_labels=sorted(clientes.keys(), key=lambda k: clientes[k][0]))

M = construir()

def esperada(producto, canal, cli_prom, cli_frec):
    row=M["pnum"].loc[producto].to_dict(); row.update(M["pcat"].loc[producto].to_dict())
    row["canalventa"]=canal; row["cliente_prom_hist"]=cli_prom; row["cliente_frecuencia_hist"]=cli_frec
    row["tienda_prom_hist"]=cli_prom if canal=="Tienda" else M["g"]
    row["producto"]=producto; row["subcategoria"]=M["prod2sub"][producto]
    R=M["pre"].transform(pd.DataFrame([row])).astype("float32")
    ip=np.array([M["vocab"]["producto"].get(producto,0)],dtype="int32")
    isub=np.array([M["vocab"]["subcategoria"].get(row["subcategoria"],0)],dtype="int32")
    return max(0.0,float(M["modelo"].predict({"producto":ip,"subcategoria":isub,"resto":R},verbose=0)[0,0]))

def fig_factores(producto, canal, cli_prom, cli_frec):
    base=esperada(producto,canal,cli_prom,cli_frec)
    ef_cli=base-esperada(producto,canal,M["g"],cli_frec)
    otro="Internet" if canal=="Tienda" else "Tienda"
    ef_canal=base-esperada(producto,otro,cli_prom,cli_frec)
    fac={"Historial de compra del cliente":round(ef_cli,2),"Canal de venta":round(ef_canal,2)}
    items=sorted(fac.items(),key=lambda t:abs(t[1]))
    fig,ax=plt.subplots(figsize=(6.0,2.6))
    ax.barh([k for k,_ in items],[v for _,v in items],
            color=["#1D9E75" if v>=0 else "#888780" for _,v in items])
    ax.axvline(0,color="#444",lw=.8); ax.set_title("Por qué esta cantidad esperada",fontsize=11)
    ax.tick_params(labelsize=9); fig.tight_layout(); return fig

def revisar(cliente, producto, canal, cargada):
    cli_prom, cli_frec = M["clientes"][cliente]
    esp = esperada(producto, canal, cli_prom, cli_frec)
    hi = max(esp*3 + 2, esp + 4)          # rango normal generoso (el modelo subestima los pedidos grandes)
    cargada = int(cargada or 0)
    if cargada <= hi:
        color, icono, titulo, detalle = "#1D9E75", "✓", "Dentro de lo esperado", "No requiere revisión."
    else:
        factor = cargada / max(esp, 1)
        color, icono = ("#C0392B" if factor >= 6 else "#BA7517"), "⚠"
        titulo = "Cantidad inusual — conviene revisar"
        detalle = f"{factor:.0f}× lo típico para este cliente/producto."
    html = (f"<div style='padding:14px 16px;border-radius:10px;background:{color}1A;border:1px solid {color};'>"
            f"<div style='font-size:20px;font-weight:600;color:{color};'>{icono} {titulo}</div>"
            f"<div style='margin-top:8px;font-size:15px;'>Cargado: <b>{cargada} u</b> &nbsp;·&nbsp; Esperado: <b>{esp:.1f} u</b></div>"
            f"<div style='margin-top:4px;font-size:13px;color:#666;'>{detalle}</div></div>")
    return html, fig_factores(producto, canal, cli_prom, cli_frec)

CAVEATS = ("**Cómo leer esto.** El modelo da una *cantidad esperada* como segunda opinión, no una verdad. "
           "El umbral de aviso es **generoso a propósito**: como el modelo subestima los pedidos grandes "
           "legítimos, solo marca desvíos muy grandes (típicamente errores de carga). Un aviso significa "
           "«mirá esta línea», no «está mal». La decisión es del planificador.")

with gr.Blocks(title="Control de calidad de cantidades — Grupo 12") as demo:
    gr.Markdown("## Control de calidad de cantidades en líneas de pedido")
    gr.Markdown("El modelo (DNN, P2) calcula la cantidad **esperada** y marca las líneas cargadas que se alejan mucho. "
                "**Human-in-the-Loop:** prioriza qué revisar; **vos** confirmás o corregís.")
    with gr.Row():
        with gr.Column():
            cliente=gr.Dropdown(M["cli_labels"], value=M["cli_labels"][len(M["cli_labels"])//2], label="Cliente (su historial se toma solo)")
            producto=gr.Dropdown(M["productos"], value="Mountain-200 Black, 42", label="Producto")
            canal=gr.Radio(["Internet","Tienda"], value="Tienda", label="Canal de venta")
            cargada=gr.Number(value=40, label="Cantidad cargada en el pedido (a revisar)", precision=0, interactive=True)
        with gr.Column():
            semaforo=gr.HTML()
            plot=gr.Plot(label="Explicación")
    gr.Markdown(CAVEATS)
    ent=[cliente,producto,canal,cargada]; sal=[semaforo,plot]
    for c in ent: c.change(revisar, ent, sal)
    demo.load(revisar, ent, sal)

In [4]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de1e59ac55ead0eae0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
